In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re


In [10]:
df = pd.read_csv('D:/Study/data_science/underpriced-listing-predictor/data/02.parsed/all_appliances_parsed.csv')
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nFeatures list length distribution per category:")
df['n_features'] = df['features'].str.count("',") + 1
print(df.groupby('category')['n_features'].describe())
print("\n--- Sample rows ---")
for cat in df['category'].unique():
    print(f"\n=== {cat} ===")
    print(df[df['category']==cat].iloc[0][['name', 'price', 'rating', 'features']].to_dict())

Shape: (2925, 5)
Columns: ['name', 'price', 'rating', 'category', 'features']

Features list length distribution per category:
                 count       mean       std  min   25%   50%   75%   max
category                                                                
AC              1020.0  10.945098  1.139412  5.0  10.0  11.0  12.0  15.0
Refrigerator    1020.0   8.255882  1.433585  2.0   8.0   8.0   9.0  11.0
WashingMachine   885.0  11.027119  1.348432  4.0  10.0  11.0  12.0  18.0

--- Sample rows ---

=== AC ===
{'name': 'Whirlpool SAI18B52MCD1 1.5 Ton 5 Star Inverter Split AC', 'price': '₹24,990', 'rating': '--rating: 4.65;', 'features': "['Split AC', '1.5 Ton Capacity', 'Air Swing', 'Auto Restart', 'Turbo Mode', 'Sleep Mode', 'Dust Filter', 'Self Diagnosis', 'Night Glow Buttons', '5 Star Rating']"}

=== Refrigerator ===
{'name': 'Samsung RT28C3733S8 236 L 3 Star Double Door Refrigerator', 'price': '₹25,490', 'rating': '--rating:4.3;', 'features': "['Multi Door', 'Top Mounted F

In [11]:
# Let us check for null values
print(df.isnull().sum()) # Zero null values in our data

name          0
price         0
rating        0
category      0
features      0
n_features    0
dtype: int64


- No Null values found

In [12]:
df.drop_duplicates(inplace=True)

### **UNIVERSAL COLUMNS**

### ***1. Cleaning Price Column***

In [13]:
df['price'] = pd.to_numeric(df['price'].str.replace('[₹,]', '', regex=True), errors='coerce')

print(df.info())
print(df['price'].sample(5))

# Check how many rows became NaN during coercion
nans = df['price'].isna().sum()
print(f"Price cleaning: {nans} rows coerced to NaN")

# Check for any remaining non-numeric values (sanity check)
if nans > 0:
    print("Warning: Some prices were not converted. Inspecting first few NaNs:")
    print(df[df['price'].isna()].head())

<class 'pandas.DataFrame'>
Index: 2913 entries, 0 to 2924
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   name        2913 non-null   str  
 1   price       2913 non-null   int64
 2   rating      2913 non-null   str  
 3   category    2913 non-null   str  
 4   features    2913 non-null   str  
 5   n_features  2913 non-null   int64
dtypes: int64(2), str(4)
memory usage: 902.5 KB
None
1209    24490
2742    25990
1336    10990
2210    41718
1957    17490
Name: price, dtype: int64
Price cleaning: 0 rows coerced to NaN


### ***2. Cleaning Rating Column***

In [14]:
df['rating'] = pd.to_numeric(df['rating'].str.split(':').str[1].str.strip().str.replace(';','' , regex=True),errors='coerce')
print(df.info())
print(df['rating'].sample(5))

# Check how many rows became NaN during coercion
nans = df['rating'].isna().sum()
print(f"Rating cleaning: {nans} rows coerced to NaN")

# Check for any remaining non-numeric values (sanity check)
if nans > 0:
    print("Warning: Some ratings were not converted. Inspecting first few NaNs:")
    print(df[df['rating'].isna()].head())

<class 'pandas.DataFrame'>
Index: 2913 entries, 0 to 2924
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   name        2913 non-null   str    
 1   price       2913 non-null   int64  
 2   rating      2913 non-null   float64
 3   category    2913 non-null   str    
 4   features    2913 non-null   str    
 5   n_features  2913 non-null   int64  
dtypes: float64(1), int64(2), str(3)
memory usage: 861.2 KB
None
2544    4.7
781     4.7
2676    4.6
1717    4.6
1931    4.0
Name: rating, dtype: float64
Rating cleaning: 0 rows coerced to NaN


### ***3. Cleaning Name Column***

**(i) Brand Name***

In [15]:
# 1. Extract brand name (first word)
df['brand_name'] = df['name'].str.split(" ").str[0]

# 2. Define the correction map
brand_corrections = {
    "O": "O General",
    "Blue": "Blue Star"
    # Only if they are different
    # Add the other broken brands you found here...
}

# 3. Apply the correction
df['brand_name'] = df['brand_name'].replace(brand_corrections).str.lower()

# 4. Verify the results
print("Brand name distribution:")
print(df['brand_name'].value_counts().head(10))

Brand name distribution:
brand_name
haier        314
samsung      268
lg           262
whirlpool    256
godrej       233
voltas       219
blue star    165
ifb          151
lloyd        141
panasonic    123
Name: count, dtype: int64


**(ii) Capacity Column**

In [16]:
# This assigns the two captured groups directly to new columns in your df
df[['capacity_value', 'capacity_unit']] = df['features'].str.extract(r'(\d+\.?\d*)\s*(Ton|-Ton|L|-L|Kg|-Kg)',re.IGNORECASE)

# Now convert the value column to float, because extract returns everything as strings
df['capacity_value'] = pd.to_numeric(df['capacity_value'], errors='coerce')

In [17]:
df.info()

<class 'pandas.DataFrame'>
Index: 2913 entries, 0 to 2924
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   name            2913 non-null   str    
 1   price           2913 non-null   int64  
 2   rating          2913 non-null   float64
 3   category        2913 non-null   str    
 4   features        2913 non-null   str    
 5   n_features      2913 non-null   int64  
 6   brand_name      2913 non-null   object 
 7   capacity_value  2911 non-null   float64
 8   capacity_unit   2911 non-null   str    
dtypes: float64(2), int64(2), object(1), str(4)
memory usage: 935.5+ KB


In [18]:
idx =df[(df['capacity_value'].isna())].index.to_list()

In [19]:
df.loc[idx, ['capacity_value', 'capacity_unit']] = [[2.02, 'Ton'],[30,'kg']]
df[(df['capacity_value'].isna())]


,name,price,rating,category,features,n_features,brand_name,capacity_value,capacity_unit


In [20]:
df['capacity_unit'].value_counts()

capacity_unit
Ton    1017
L      1013
kg      881
Kg        2
Name: count, dtype: int64

In [21]:
idx = df[df['capacity_unit']=='Kg'].index.to_list()
df.loc[idx,'capacity_unit']='kg'

In [22]:
df['capacity_unit'].value_counts()

capacity_unit
Ton    1017
L      1013
kg      883
Name: count, dtype: int64

***(iii) Extract Star Rating from Name***

In [23]:
# Extract star rating from name using regex - handles:
# - Digits followed by optional spaces, optional hyphen, optional spaces, then "star" (case-insensitive)
# Examples: "5 Star", "4-Star", "3-star", "2 star", etc.
df['star_rating_from_name'] = df['name'].str.extract(r'(\d+)\s*-?\s*star', expand=False, flags=re.IGNORECASE)

# Convert to numeric (handle any non-numeric values)
df['star_rating_from_name'] = pd.to_numeric(df['star_rating_from_name'], errors='coerce')


**Finding Star Ratings From Features**

In [24]:
df['star_rating_from_features'] = df['features'].str.extract(r'(\d+)\s*-?\s*star', expand=False, flags=re.IGNORECASE)
df['star_rating_from_features'] = pd.to_numeric(df['star_rating_from_features'], errors='coerce')

In [25]:
# 3. Fill NaNs in the main column using the features-extracted values
df['star_rating'] = df['star_rating_from_name'].fillna(df['star_rating_from_features'])

In [26]:
# Add binary indicator for categories that typically have energy star ratings
df['has_star_rating'] = df['category'].isin(['AC', 'Refrigerator']).astype(int)


In [27]:
idx = df[df['category']=='WashingMachine'].index.to_list()
df.loc[idx,'star_rating'] = np.nan
# filled all the star ratings of WashingMachine as nan 

In [28]:
# drop the temporary columns
del df['star_rating_from_features'] ,df['star_rating_from_name']

In [29]:
df.info()

<class 'pandas.DataFrame'>
Index: 2913 entries, 0 to 2924
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             2913 non-null   str    
 1   price            2913 non-null   int64  
 2   rating           2913 non-null   float64
 3   category         2913 non-null   str    
 4   features         2913 non-null   str    
 5   n_features       2913 non-null   int64  
 6   brand_name       2913 non-null   object 
 7   capacity_value   2913 non-null   float64
 8   capacity_unit    2913 non-null   str    
 9   star_rating      1947 non-null   float64
 10  has_star_rating  2913 non-null   int64  
dtypes: float64(3), int64(3), object(1), str(4)
memory usage: 1.0+ MB


### **UNIVERSAL FEATURES**

In [30]:
# has_inverter
df['has_inverter'] = df['features'].str.contains(r'Inverter',case=False).astype(int)

In [31]:
# has_wifi, has_app_control, has_voice_control
df['has_wifi'] = df['features'].str.contains(r'Wi-fi',case=False).astype(int)
df['has_voice_control'] = df['features'].str.contains(r'Voice',case=False).astype(int)
df['has_app_control'] = df['features'].str.contains(r'APP',case=False).astype(int)


In [32]:
# smart_connectivity_score (0 to 3) Total
df['smart_connectivity_score'] = df['has_wifi']+df['has_voice_control']+df['has_app_control']

In [33]:
# model_year
df['model_year'] = df['name'].str.extract(r'\b(201[0-9]|202[0-9])\b').astype('Int64')

# is_recent_model
df['is_recent_model'] = np.where(
    df['model_year'].isna(),
    np.nan,
    (df['model_year'] >= 2024)
)
df['is_recent_model'] = df['is_recent_model'].astype('Int8')

# appliance_age

df['appliance_age'] = df['model_year'].apply(lambda x: 2026 - x).astype('Int8')

### ***AC_Specific Columns***

In [34]:
# Split AC and Window AC binary column. Also remove ACs other than Split and Windows.

ac_df = df[df['category']=='AC']
df['ac_split'] = ac_df['features'].str.contains(r'Split' , re.IGNORECASE).astype('Int8')
df['ac_window'] = ac_df['features'].str.contains(r'Window' , re.IGNORECASE).astype('Int8')

idx = ac_df[~(ac_df['features'].str.contains(r'Split|Window'))].index.to_list() # indices of AC otherthan Split|window
df = df.drop(idx, errors='ignore')


In [35]:
# has pm25 filter
df['ac_pm25_filter'] = ac_df['features'].str.contains(
    r'pm\s*2\.?5',
    case=False,
    na=False,
    regex=True
)
df['ac_pm25_filter'] = df['ac_pm25_filter'].astype('Int8')

# has hepa filter
df['ac_hepa_filter'] = ac_df['features'].str.contains(
    r'hepa',
    case=False,
    na=False,
    regex=True
)
df['ac_hepa_filter'] = df['ac_hepa_filter'].astype('Int8')

# has auto clean
df['ac_auto_clean'] = ac_df['features'].str.contains(
    r'auto clean',
    case=False,
    na=False,
    regex=True
)
df['ac_auto_clean'] = df['ac_auto_clean'].astype('Int8')

# has Hot and Cold
df['ac_hot_and_cold'] = ac_df['features'].str.contains(
    r'hot|cold',
    case=False,
    na=False,
    regex=True
)

df['ac_hot_and_cold'] = df['ac_hot_and_cold'].astype('Int8')

# has Copper condenser
df['ac_copper_condenser'] = ac_df['features'].str.contains(
    r'copper|condenser',
    case=False,
    na=False,
    regex=True
)

df['ac_copper_condenser'] = df['ac_copper_condenser'].astype('Int8')




In [36]:
df.info()

<class 'pandas.DataFrame'>
Index: 2905 entries, 0 to 2924
Data columns (total 26 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   name                      2905 non-null   str    
 1   price                     2905 non-null   int64  
 2   rating                    2905 non-null   float64
 3   category                  2905 non-null   str    
 4   features                  2905 non-null   str    
 5   n_features                2905 non-null   int64  
 6   brand_name                2905 non-null   object 
 7   capacity_value            2905 non-null   float64
 8   capacity_unit             2905 non-null   str    
 9   star_rating               1939 non-null   float64
 10  has_star_rating           2905 non-null   int64  
 11  has_inverter              2905 non-null   int64  
 12  has_wifi                  2905 non-null   int64  
 13  has_voice_control         2905 non-null   int64  
 14  has_app_control         

In [37]:
df['ac_copper_condenser'].sample(10)

2720    <NA>
2241    <NA>
133        0
1821    <NA>
2863    <NA>
1505    <NA>
2157    <NA>
301        0
2456    <NA>
482        1
Name: ac_copper_condenser, dtype: Int8

In [38]:
# Apply the counter to check the 10 most common Features
from collections import Counter

all_features = []
ac_df = df[df['category']=='AC'].copy()

import ast

ac_df['features'] = ac_df['features'].apply(ast.literal_eval) # This converts dtype of features column into list

for row in ac_df['features']:
    all_features.extend(row)

best_features = Counter(all_features).most_common(10)
binary_feat = []
for i in best_features:
    binary_feat.append(i[0])

print(binary_feat)

['Dust Filter', 'Auto Restart', 'Sleep Mode', 'Split AC', 'Dehumidification', 'Inverter', 'Turbo Mode', '1.5 Ton Capacity', 'Air Swing', 'Self Diagnosis']


In [39]:
remove_features = [
    'Dust Filter', 'Auto Restart', 'Sleep Mode', 'Split AC',
    'Inverter','1.5 Ton Capacity', 'Air Swing'
]

binary_feat = [feat for feat in binary_feat if feat not in remove_features]

print(binary_feat)

['Dehumidification', 'Turbo Mode', 'Self Diagnosis']


In [40]:
for feat in binary_feat:
    df[f'ac_{feat}'] = ac_df['features'].apply(lambda x : int(feat in x))
    df[f'ac_{feat}'] = df[f'ac_{feat}'].astype('Int8')

print(df.info())


<class 'pandas.DataFrame'>
Index: 2905 entries, 0 to 2924
Data columns (total 29 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   name                      2905 non-null   str    
 1   price                     2905 non-null   int64  
 2   rating                    2905 non-null   float64
 3   category                  2905 non-null   str    
 4   features                  2905 non-null   str    
 5   n_features                2905 non-null   int64  
 6   brand_name                2905 non-null   object 
 7   capacity_value            2905 non-null   float64
 8   capacity_unit             2905 non-null   str    
 9   star_rating               1939 non-null   float64
 10  has_star_rating           2905 non-null   int64  
 11  has_inverter              2905 non-null   int64  
 12  has_wifi                  2905 non-null   int64  
 13  has_voice_control         2905 non-null   int64  
 14  has_app_control         

### ***Refrigerator Based Columns***

In [41]:
fridge_df = df[df['category']=='Refrigerator']

fridge_df.shape

(1013, 29)

In [42]:
fridge_df['features'].str.split(",").str[0].str.strip("[]'").str.strip().value_counts()

features
Single Door       453
Multi Door        376
Chest Freezer      94
Side By Side       67
French Door        18
Visi Cooler         3
339 L Capacity      1
370 L Capacity      1
Name: count, dtype: int64

We have Single Door, Multi Door, Chest Freezer, Side By Side, French.
Rest We will drop.

In [43]:
df['ref_single_door'] = fridge_df['features'].str.contains(r'Single',re.IGNORECASE).astype('Int8')
df['ref_multi_door'] = fridge_df['features'].str.contains(r'Multi',re.IGNORECASE).astype('Int8')
# we have double and triple inside multi doors

df['ref_chest_freezer'] = fridge_df['features'].str.contains(r'Chest',re.IGNORECASE).astype('Int8')
df['ref_side_door'] = fridge_df['features'].str.contains(r'Side',re.IGNORECASE).astype('Int8')
df['ref_french_door'] = fridge_df['features'].str.contains(r'French',re.IGNORECASE).astype('Int8')


idx = fridge_df[~fridge_df['features'].str.contains(r'Single|Multi|Chest|Side|French')].index.to_list()
df.drop(idx , inplace=True)

In [44]:
# double door refrigerators

df['ref_double_door'] = (
    fridge_df['name'].str.contains(r'Double', case=False, na=False)& (df['ref_multi_door'] == 1)
).astype('Int8')

# Triple door refrigerators
df['ref_triple_door'] = (
    fridge_df['name'].str.contains(r'Triple', case=False, na=False)& (df['ref_multi_door'] == 1)
).astype('Int8')

In [45]:
df.info()

<class 'pandas.DataFrame'>
Index: 2900 entries, 0 to 2924
Data columns (total 36 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   name                      2900 non-null   str    
 1   price                     2900 non-null   int64  
 2   rating                    2900 non-null   float64
 3   category                  2900 non-null   str    
 4   features                  2900 non-null   str    
 5   n_features                2900 non-null   int64  
 6   brand_name                2900 non-null   object 
 7   capacity_value            2900 non-null   float64
 8   capacity_unit             2900 non-null   str    
 9   star_rating               1936 non-null   float64
 10  has_star_rating           2900 non-null   int64  
 11  has_inverter              2900 non-null   int64  
 12  has_wifi                  2900 non-null   int64  
 13  has_voice_control         2900 non-null   int64  
 14  has_app_control         

In [46]:
# frost free fridges

df['ref_frost_free'] = (
    fridge_df['features']
    .str.contains(r'Frost Free', case=False,na=False)
    .astype('Int8')
)

# Convertible fridges
df['ref_convertible'] = (
    fridge_df['features']
    .str.contains(r'Convertible', case=False,na=False)
    .astype('Int8')
)

# Door Alarm fridges
df['ref_door_alarm'] = (
    fridge_df['features']
    .str.contains(r'Door alarm', case=False,na=False)
    .astype('Int8')
)

# Door Lock fridges
df['ref_door_lock'] = (
    fridge_df['features']
    .str.contains(r'Child lock|Door Lock|lock', case=False,na=False)
    .astype('Int8')
)

# Dispenser fridges
df['ref_dispenser'] = (
    fridge_df['features']
    .str.contains(r'Dispenser', case=False,na=False)
    .astype('Int8')
)

# Door Display fridges
df['ref_door_display'] = (
    fridge_df['features']
    .str.contains(r'Display', case=False,na=False)
    .astype('Int8')
)

# Mini Refrigerators

df['ref_mini'] = (
    fridge_df['name']
    .str.contains(r'\bmini\b', case=False, na=False)
    .astype('Int8')
)



In [47]:
df.info()

<class 'pandas.DataFrame'>
Index: 2900 entries, 0 to 2924
Data columns (total 43 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   name                      2900 non-null   str    
 1   price                     2900 non-null   int64  
 2   rating                    2900 non-null   float64
 3   category                  2900 non-null   str    
 4   features                  2900 non-null   str    
 5   n_features                2900 non-null   int64  
 6   brand_name                2900 non-null   object 
 7   capacity_value            2900 non-null   float64
 8   capacity_unit             2900 non-null   str    
 9   star_rating               1936 non-null   float64
 10  has_star_rating           2900 non-null   int64  
 11  has_inverter              2900 non-null   int64  
 12  has_wifi                  2900 non-null   int64  
 13  has_voice_control         2900 non-null   int64  
 14  has_app_control         

### ***Washing Machines***

In [48]:
wm_df = df[df['category']=='WashingMachine']

In [49]:
wm_df['features'].str.contains(r'automatic',case=False,na=False)

2040    True
2041    True
2042    True
2043    True
2044    True
        ... 
2920    True
2921    True
2922    True
2923    True
2924    True
Name: features, Length: 883, dtype: bool

In [50]:
# semi automatic & fully automatic
df['wm_fully_automatic'] = (wm_df['features'].str.contains(r'Fully' , case=False,na=False).astype('Int8'))
df['wm_semi_automatic'] = (wm_df['features'].str.contains(r'Semi' , case=False,na=False).astype('Int8'))

# Washer Only and Dryer Only Machines and both 
df['wm_with_dryer'] = (wm_df['features'].str.contains(r'Washing Machine with dryer',case=False,na=False).astype('Int8'))
df['wm_washer_only'] = (wm_df['name'].str.contains(r'Washer only',case=False,na=False).astype('Int8'))
df['wm_dryer_only'] = (~wm_df['name'].str.contains(r'wash', case=False, na=False)).astype('Int8')


In [51]:
~(wm_df['name'].str.contains(r'wash',case=False,na=False))

2040    False
2041    False
2042    False
2043    False
2044    False
        ...  
2920    False
2921    False
2922    False
2923    False
2924    False
Name: name, Length: 883, dtype: bool

In [52]:
# Washing Machines with Front and Top Load
df['wm_front_load'] = (wm_df['features'].str.contains(r'Front load',case=False,na=False).astype('Int8'))
df['wm_top_load'] = (wm_df['features'].str.contains(r'top load',case=False,na=False).astype('Int8'))

# has inbuilt heater, has quick wash, has stainless steel tub, child lock, shockProof, display
df['wm_inbuilt_heater'] = (wm_df['features'].str.contains(r'Heater',case=False,na=False).astype('Int8'))
df['wm_quick_wash'] = (wm_df['features'].str.contains(r'quick wash',case=False,na=False).astype('Int8'))
df['wm_ss_tub'] = (wm_df['features'].str.contains(r'stainless steel',case=False,na=False).astype('Int8'))
df['wm_child_lock'] = (wm_df['features'].str.contains(r'child lock',case=False,na=False).astype('Int8'))
df['wm_shock_proof'] = (wm_df['features'].str.contains(r'shock proof',case=False,na=False).astype('Int8'))
df['wm_display'] = (wm_df['features'].str.contains(r'digital display|led display|lcd display',case=False,na=False).astype('Int8'))



In [53]:
df.info()

<class 'pandas.DataFrame'>
Index: 2900 entries, 0 to 2924
Data columns (total 56 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   name                      2900 non-null   str    
 1   price                     2900 non-null   int64  
 2   rating                    2900 non-null   float64
 3   category                  2900 non-null   str    
 4   features                  2900 non-null   str    
 5   n_features                2900 non-null   int64  
 6   brand_name                2900 non-null   object 
 7   capacity_value            2900 non-null   float64
 8   capacity_unit             2900 non-null   str    
 9   star_rating               1936 non-null   float64
 10  has_star_rating           2900 non-null   int64  
 11  has_inverter              2900 non-null   int64  
 12  has_wifi                  2900 non-null   int64  
 13  has_voice_control         2900 non-null   int64  
 14  has_app_control         

In [54]:
# dropping name and Feature column since we have extracted all the data from it !!
df.drop(columns=['name','features'], inplace=True)

In [60]:
df.groupby('brand_name')['price'].median().sort_values(ascending=False)

brand_name
mitsubishi    56240.0
o general     51245.0
electrolux    49900.0
siemens       42990.0
daikin        40023.0
panasonic     39000.0
toshiba       37990.0
hitachi       37650.0
imee          37597.0
blue star     37500.0
carrier       37495.0
lg            37490.0
bosch         35680.0
western       33500.0
ifb           33500.0
samsung       33490.0
hisense       32490.0
cruise        31990.0
lloyd         31190.0
rockwell      30599.0
croma         29490.0
bpl           29019.5
candy         28990.0
tcl           27990.0
godrej        27929.0
liebherr      27290.0
haier         27124.0
hafele        26990.0
voltas        26094.5
black         25994.5
motorola      23290.0
equator       22331.0
midea         21990.0
sharp         21749.0
kenstar       21490.0
whirlpool     20590.0
realme        19790.0
sansui        17490.0
kelvinator    17410.5
acer          16500.0
onida         15690.0
intex         12999.0
minifrost     12990.0
ionstar       12990.0
singer        12690.0

In [59]:
df['brand_name'] = df['brand_name'].replace({'elextrolux':'electrolux'})

In [57]:
df.to_parquet('D:/Study/data_science/underpriced-listing-predictor/data/03.cleaned/multi_appliances_cleaned.parquet')
df.to_csv('D:/Study/data_science/underpriced-listing-predictor/data/03.cleaned/multi_appliances_cleaned.csv')